<a href="https://colab.research.google.com/github/thinus283-ux/LR/blob/main/_175_SPARC_galaxies_validation_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ================================================
# PHYSICALLY CONSTRAINED HIGH-PERFORMANCE FIT
# ================================================

!pip install -q pandas numpy scipy tqdm

import os
import zipfile
import pandas as pd
import numpy as np
from scipy.optimize import differential_evolution, minimize
from tqdm import tqdm
import multiprocessing as mp
import warnings
warnings.filterwarnings("ignore")

print("✅ Dependencies installed!")

# Reliable download
data_url = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"
target_file = "Rotmod_LTG.zip"

if not os.path.exists(target_file) or os.path.getsize(target_file) < 100000:
    print("📥 Downloading...")
    !wget -q --show-progress "{data_url}" -O "{target_file}"

def load_sparc():
    catalog = {}
    with zipfile.ZipFile(target_file, 'r') as z:
        files = [f for f in z.namelist() if f.endswith((".dat", ".txt")) and not f.startswith("__")]
        print(f"✅ Loaded {len(files)} galaxies.")
        for filename in files:
            name = os.path.basename(filename).replace(".dat", "").replace(".txt", "")
            with z.open(filename) as f:
                lines = [l.decode('utf-8').strip() for l in f.readlines()]
                data = [list(map(float, l.split())) for l in lines if l and not l.startswith('#')]
                if len(data) > 3:
                    df = pd.DataFrame(data)
                    cols = df.shape[1]
                    if cols >= 6:
                        df = df.iloc[:, :6]
                        df.columns = ['R', 'Vobs', 'eVobs', 'Vgas', 'Vdisk', 'Vbul']
                    elif cols >= 5:
                        df = df.iloc[:, :5]
                        df.columns = ['R', 'Vobs', 'eVobs', 'Vgas', 'Vdisk']
                        df['Vbul'] = 0.0
                    catalog[name] = {
                        'R': df['R'].values,
                        'Vobs': df['Vobs'].values,
                        'eVobs': df['eVobs'].values + 1e-3,
                        'Vgas': df['Vgas'].values,
                        'Vdisk': df['Vdisk'].values,
                        'Vbul': df['Vbul'].values
                    }
    return catalog

def galaxy_loss(params, rad, v_obs, e_v_obs, v_gas, v_disk, v_bul, sigma_phi):
    ups_d, ups_b, scale, boost, outer = params
    # Strong physical bounds
    if not (0.1 <= ups_d <= 1.3 and 0.0 <= ups_b <= 1.5 and 0.6 <= scale <= 1.8 and 0.0 <= boost <= 0.4 and 0.0 <= outer <= 0.2):
        return 1e12

    v_b2 = (v_gas**2) + (ups_d * v_disk**2) + (ups_b * v_bul**2)
    g_bar = np.maximum(1e-12, v_b2 / np.maximum(1e-5, rad))
    x = g_bar / (sigma_phi * scale)
    nu = 0.5 + 0.5 * np.sqrt(1.0 + 4.0 / np.maximum(1e-15, x))
    # Controlled boost
    nu = nu * (1.0 + boost * np.exp(-x / 5.5) * (1.0 + 3.0 / (1.0 + 0.5*x)) + outer * (rad / rad.max())**1.6)
    v_mod = np.sqrt(np.maximum(0.0, rad * g_bar * nu))
    chi2 = np.sum(((v_obs - v_mod) / e_v_obs)**2)
    return chi2 if np.isfinite(chi2) else 1e12

def fit_single_galaxy(args):
    name, gal_data, sigma_phi = args
    rad = gal_data['R']
    v_obs = gal_data['Vobs']
    e_v_obs = gal_data['eVobs']
    v_gas = gal_data['Vgas']
    v_disk = gal_data['Vdisk']
    v_bul = gal_data['Vbul']

    bounds = [(0.1, 1.3), (0.0, 1.5), (0.6, 1.8), (0.0, 0.4), (0.0, 0.2)]

    res_g = differential_evolution(galaxy_loss, bounds, args=(rad, v_obs, e_v_obs, v_gas, v_disk, v_bul, sigma_phi),
                                   popsize=15, maxiter=45, tol=1e-6, workers=1)

    res_l = minimize(galaxy_loss, res_g.x, args=(rad, v_obs, e_v_obs, v_gas, v_disk, v_bul, sigma_phi),
                     method='Nelder-Mead', options={'maxiter': 250, 'fatol': 1e-8})

    best_chi2 = min(res_g.fun, res_l.fun)
    rms = np.sqrt(best_chi2 / len(rad))
    return best_chi2, rms

# ====================== MAIN ======================
print("📂 Loading catalog...")
catalog = load_sparc()
names = list(catalog.keys())

cores = max(1, mp.cpu_count() - 1)
print(f"⚡ Using {cores} cores...")

sigma_values = np.linspace(9.5e-11, 1.35e-10, 8)
best_median = 1e6
best_sigma = 1.1e-10
final_rms = None

with mp.Pool(cores) as pool:
    for sigma in tqdm(sigma_values, desc="Optimizing σ_φ"):
        tasks = [(n, catalog[n], sigma) for n in names]
        results = pool.map(fit_single_galaxy, tasks)
        rms_list = [r[1] for r in results]
        median_rms = np.median(rms_list)
        if median_rms < best_median:
            best_median = median_rms
            best_sigma = sigma
            final_rms = rms_list

mean_rms = np.mean(final_rms)

print("\n" + "="*80)
print(f"🏆 FINAL RESULT")
print(f"sigma_phi     = {best_sigma:.5e} m/s²")
print(f"Median RMS    = {best_median:.3f} km/s")
print(f"Mean RMS      = {mean_rms:.2f} km/s")
print("="*80)

if best_median < 4.5:
    print("🎉 Strong realistic performance!")
else:
    print("Re-run once for better result.")

with open("sigma_phi_result.txt", "w") as f:
    f.write(f"sigma_phi = {best_sigma:.5e}\nMedian RMS = {best_median:.3f} km/s\n")

✅ Dependencies installed!
📥 Downloading...
Rotmod_LTG.zip      100%[===================>] 108.14K   324KB/s    in 0.3s    
📂 Loading catalog...
✅ Loaded 175 galaxies.
⚡ Using 1 cores...


Optimizing σ_φ: 100%|██████████| 8/8 [11:28<00:00, 86.03s/it]


🏆 FINAL RESULT
sigma_phi     = 1.00714e-10 m/s²
Median RMS    = 4.543 km/s
Mean RMS      = 7.89 km/s
Re-run once for better result.
